# Pelt with the compiled C++ backend

<!-- {{ add_binder_block(page) }} -->

With the built-in `l2` cost, `min_size=1` and `jump=1` and a positive penalty, the `Pelt` estimator transparently delegates the segmentation to a compiled C++ implementation of the textbook PELT algorithm (`ruptures._core`, imported from the *biblio-these* project) — much faster, same breakpoints. The backend is optional: when the extension is not built, or for any other configuration, `Pelt` uses the pure Python implementation. The usage does not change.

## Setup

In [ ]:
import time
from unittest import mock

import ruptures as rpt

# The compiled backend is a private, optional extension.
try:
    from ruptures import _core

    print("compiled backend: available")
except ImportError:
    print("compiled backend: not built; Pelt uses the Python implementation")

## Minimal usage

Nothing to do but use `Pelt` as usual; the delegation happens automatically.

In [ ]:
signal, true_bkps = rpt.pw_constant(500, 1, 3, noise_std=1, seed=42)

algo = rpt.Pelt(model="l2", min_size=1, jump=1)
bkps = algo.fit_predict(signal, pen=50.0)  # delegated to C++ when available
print(bkps)

## Any other configuration runs the Python implementation

Only the configuration that the compiled algorithm covers is delegated; everything else is unchanged.

In [ ]:
bkps = rpt.Pelt(model="l2").fit_predict(signal, pen=50.0)  # default min_size=2, jump=5
print(bkps)

## What the delegation buys

Timings and breakpoints, for the compiled and the pure Python paths.

In [ ]:
# compiled path
t0 = time.perf_counter()
compiled_bkps = algo.predict(pen=50.0)
t1 = time.perf_counter()

# pure Python path, obtained by disabling the backend
with mock.patch("ruptures.detection.pelt._core", None):
    python_bkps = algo.predict(pen=50.0)
t2 = time.perf_counter()

print(f"compiled: {(t1 - t0) * 1e3:.1f} ms")
print(f"python:   {(t2 - t1) * 1e3:.1f} ms")
print("identical breakpoints:", compiled_bkps == python_bkps)